# Adaptive vs. Fixed FMM Benchmark

This notebook compares the current **adaptive-order / error-aware acceptance model**
against a **fixed-order baseline** in `jaccpot`.

It is split into two parts:

1. **Performance**: warmed runtime comparisons with a tree / traversal / M2L / total split.
2. **Accuracy**: direct-sum comparisons with Dehnen-style diagnostics based on per-particle
   relative acceleration errors and error distributions.

The current comparison uses the real-basis `accurate` preset and contrasts:

- `adaptive_order=False`, `max_order=4`
- `adaptive_order=True`, `p_gears=(2, 3, 4)`

The adaptive path uses the current solver-owned acceptance model:

- highest available order decides whether a pair is safe to accept,
- a relaxed geometric guard prevents pathological over-acceptance,
- the first passing order chooses the far-field work bucket.


In [ ]:
import math
from dataclasses import dataclass

import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np

from examples.benchmark_utils import generate_random_distribution, time_callable
from jaccpot import FastMultipoleMethod
from jaccpot.runtime._adaptive_policy import (
    adaptive_pair_policy,
    adaptive_policy_tolerance,
    bucket_far_pairs_by_tag,
    dehnen_like_pair_error_by_order_from_degree_power,
    source_error_proxy_by_order_from_degree_power,
    source_power_by_degree_from_multipoles,
)
from jaccpot.runtime._interaction_cache import _build_dual_tree_artifacts
from jaccpot.runtime.reference import direct_sum as reference_direct_sum

print('jax:', jax.__version__)
print('device:', jax.devices()[0])


## Benchmark Helpers

The stage-timing helpers below mirror the runtime split used by `bench/bench_fmm.py`:

- tree / upward preparation
- traversal construction
- M2L / downward preparation
- full end-to-end acceleration evaluation

The direct-sum helpers are only used for the smaller accuracy problem sizes because
that baseline scales as `O(N^2)`.


In [ ]:
@dataclass(frozen=True)
class StageArtifacts:
    tree_artifacts: object
    runtime_overrides: object
    positions_arr: jax.Array
    masses_arr: jax.Array


def make_solver(*, adaptive_order: bool, theta: float, p_gears=(2, 3, 4),
                preset='accurate', basis='real', softening=1.0e-2,
                mac_force_scale_mode='prev', adaptive_error_model='tail_proxy',
                adaptive_eps=None, advanced=None):
    return FastMultipoleMethod(
        preset=preset,
        basis=basis,
        theta=theta,
        softening=softening,
        adaptive_order=adaptive_order,
        p_gears=tuple(int(v) for v in p_gears),
        mac_force_scale_mode=mac_force_scale_mode,
        adaptive_error_model=adaptive_error_model,
        adaptive_eps=adaptive_eps,
        advanced=advanced,
    )


def build_tree_and_upward(fmm, positions, masses, *, leaf_size, max_order):
    impl = fmm._impl
    positions_arr, masses_arr, _ = impl._prepare_state_input_arrays(positions, masses)
    runtime_overrides = impl._resolve_runtime_execution_overrides(
        num_particles=int(positions_arr.shape[0])
    )
    tree_artifacts = impl._prepare_state_tree_and_upward(
        positions_arr=positions_arr,
        masses_arr=masses_arr,
        bounds=None,
        leaf_size=int(leaf_size),
        max_order=int(max_order),
        refine_local_val=impl.refine_local,
        max_refine_levels_val=impl.max_refine_levels,
        aspect_threshold_val=impl.aspect_threshold,
        jit_tree_override=None,
        upward_center_mode=runtime_overrides.center_mode,
        allow_stateful_cache=False,
    )
    return StageArtifacts(
        tree_artifacts=tree_artifacts,
        runtime_overrides=runtime_overrides,
        positions_arr=positions_arr,
        masses_arr=masses_arr,
    )


def build_traversal(fmm, staged, *, theta):
    impl = fmm._impl
    tree_artifacts = staged.tree_artifacts
    runtime_overrides = staged.runtime_overrides
    pair_policy = None
    policy_state = None
    if impl.adaptive_order:
        pair_policy = adaptive_pair_policy
        dtype = tree_artifacts.positions_sorted.dtype
        policy_state = impl._build_adaptive_policy_state(
            upward=tree_artifacts.upward,
            p_gears=impl.p_gears,
            force_scale_nodes=jnp.ones(
                (tree_artifacts.tree.parent.shape[0],),
                dtype=dtype,
            ),
            eps=jnp.asarray(
                impl.adaptive_eps
                if getattr(impl, 'adaptive_eps', None) is not None
                else adaptive_policy_tolerance(
                    theta=float(theta),
                    p_gears=impl.p_gears,
                    dtype=dtype,
                ),
                dtype=dtype,
            ),
            theta=jnp.asarray(float(theta), dtype=dtype),
            error_model_code=jnp.asarray(
                1 if getattr(impl, 'adaptive_error_model', 'tail_proxy') == 'dehnen_degree' else 0,
                dtype=jnp.int32,
            ),
        )
    dual_artifacts, _ = _build_dual_tree_artifacts(
        tree_artifacts.tree,
        tree_artifacts.upward.geometry,
        theta=float(theta),
        mac_type=impl.mac_type,
        dehnen_radius_scale=impl.dehnen_radius_scale,
        cache_key=None,
        cache_entry=None,
        max_pair_queue=impl.max_pair_queue,
        pair_process_block=impl.pair_process_block,
        traversal_config=runtime_overrides.traversal_config,
        retry_logger=None,
        use_dense_interactions=impl.use_dense_interactions,
        grouped_interactions=runtime_overrides.grouped_interactions,
        grouped_chunk_size=runtime_overrides.m2l_chunk_size,
        pair_policy=pair_policy,
        policy_state=policy_state,
    )
    return dual_artifacts


def fresh_locals_template(fmm, staged, *, max_order):
    impl = fmm._impl
    tree_artifacts = staged.tree_artifacts
    return impl._build_locals_template_for_prepare_state(
        tree=tree_artifacts.tree,
        upward=tree_artifacts.upward,
        max_order=int(max_order),
        pos_sorted=tree_artifacts.positions_sorted,
    )


def adaptive_far_pairs_by_gear(fmm, dual_artifacts):
    if not fmm._impl.adaptive_order:
        return None
    traversal_result = dual_artifacts.traversal_result
    far_total = int(traversal_result.far_pair_count)
    return bucket_far_pairs_by_tag(
        jnp.asarray(traversal_result.interaction_sources[:far_total], dtype=jnp.int32),
        jnp.asarray(traversal_result.interaction_targets[:far_total], dtype=jnp.int32),
        jnp.asarray(traversal_result.interaction_tags[:far_total], dtype=jnp.int32),
        num_tags=len(fmm._impl.p_gears),
    )


def run_m2l_downward(fmm, staged, dual_artifacts, *, theta, max_order):
    impl = fmm._impl
    tree_artifacts = staged.tree_artifacts
    runtime_overrides = staged.runtime_overrides

    (
        interactions,
        _neighbor_list,
        _traversal_result,
        dense_buffers,
        grouped_buffers,
        grouped_segment_starts,
        grouped_segment_lengths,
        grouped_segment_class_ids,
        grouped_segment_sort_permutation,
        grouped_segment_group_ids,
        grouped_segment_unique_targets,
    ) = impl._unpack_dual_tree_artifacts(dual_artifacts)

    locals_template = fresh_locals_template(fmm, staged, max_order=max_order)
    far_pairs_by_gear = adaptive_far_pairs_by_gear(fmm, dual_artifacts)

    return impl._prepare_downward_with_artifacts(
        tree=tree_artifacts.tree,
        upward=tree_artifacts.upward,
        theta_val=float(theta),
        locals_template=locals_template,
        interactions=interactions,
        runtime_m2l_chunk_size=runtime_overrides.m2l_chunk_size,
        runtime_l2l_chunk_size=runtime_overrides.l2l_chunk_size,
        runtime_traversal_config=runtime_overrides.traversal_config,
        record_retry=lambda _: None,
        dense_buffers=dense_buffers,
        grouped_interactions=runtime_overrides.grouped_interactions,
        grouped_buffers=grouped_buffers,
        grouped_segment_starts=grouped_segment_starts,
        grouped_segment_lengths=grouped_segment_lengths,
        grouped_segment_class_ids=grouped_segment_class_ids,
        grouped_segment_sort_permutation=grouped_segment_sort_permutation,
        grouped_segment_group_ids=grouped_segment_group_ids,
        grouped_segment_unique_targets=grouped_segment_unique_targets,
        farfield_mode=runtime_overrides.farfield_mode,
        far_pairs_by_gear=far_pairs_by_gear,
        adaptive_order=impl.adaptive_order,
        p_gears=impl.p_gears,
    )


def benchmark_runtime_case(*, n, theta=0.5, leaf_size=16, max_order=4,
                           warmup=3, runs=2, dtype=jnp.float64,
                           p_gears=(2, 3, 4), seed=0):
    key = jax.random.PRNGKey(seed)
    positions, masses, _ = generate_random_distribution(
        n,
        key=key,
        dtype=dtype,
    )
    results = {}
    for label, adaptive in [('fixed', False), ('adaptive', True)]:
        solver = make_solver(
            adaptive_order=adaptive,
            theta=theta,
            p_gears=p_gears,
        )
        tree_timing = time_callable(
            build_tree_and_upward,
            solver,
            positions,
            masses,
            leaf_size=leaf_size,
            max_order=max_order,
            warmup=warmup,
            runs=runs,
        )
        staged = tree_timing.result
        traversal_timing = time_callable(
            build_traversal,
            solver,
            staged,
            theta=theta,
            warmup=warmup,
            runs=runs,
        )
        dual_artifacts = traversal_timing.result
        m2l_timing = time_callable(
            run_m2l_downward,
            solver,
            staged,
            dual_artifacts,
            theta=theta,
            max_order=max_order,
            warmup=warmup,
            runs=runs,
        )
        total_timing = time_callable(
            lambda pos, mass: solver.compute_accelerations(
                pos,
                mass,
                leaf_size=leaf_size,
                max_order=max_order,
            ),
            positions,
            masses,
            warmup=warmup,
            runs=runs,
        )
        traversal_result = dual_artifacts.traversal_result
        gear_counts = tuple(int(v) for v in getattr(solver._impl, '_recent_far_pairs_by_gear_counts', tuple()))
        results[label] = {
            'tree_s': tree_timing.mean,
            'traversal_s': traversal_timing.mean,
            'm2l_s': m2l_timing.mean,
            'total_s': total_timing.mean,
            'far_pairs': int(traversal_result.far_pair_count),
            'near_pairs': int(traversal_result.near_pair_count),
            'gear_counts': gear_counts,
        }
    return results


def direct_sum_accelerations(positions, masses, *, softening=1.0e-2, G=1.0):
    return jax.vmap(
        lambda x: reference_direct_sum(
            positions,
            masses,
            x,
            G=G,
            softening=softening,
        )
    )(positions)


def relative_acceleration_errors(acc, ref, *, eps=1.0e-12):
    acc = jnp.asarray(acc)
    ref = jnp.asarray(ref)
    abs_err = jnp.linalg.norm(acc - ref, axis=1)
    ref_mag = jnp.linalg.norm(ref, axis=1)
    rel_err = abs_err / jnp.maximum(ref_mag, eps)
    return rel_err, abs_err, ref_mag


def summarize_errors(rel_err):
    rel_np = np.asarray(rel_err)
    return {
        'median': float(np.median(rel_np)),
        'p90': float(np.quantile(rel_np, 0.90)),
        'p99': float(np.quantile(rel_np, 0.99)),
        'max': float(np.max(rel_np)),
    }


def empirical_cdf(values):
    arr = np.sort(np.asarray(values))
    frac = (np.arange(arr.size) + 1) / arr.size
    return arr, frac


## Runtime Benchmark

The next cell runs a warmed benchmark on a small list of `N` values and reports:

- tree/upward time
- traversal time
- M2L/downward preparation time
- total acceleration-evaluation time
- far/near interaction counts
- adaptive gear counts

These timings are intended to compare **fixed** and **adaptive** criteria on the
same particle realization and solver settings.


In [ ]:
theta = 0.5
leaf_size = 16
max_order = 4
p_gears = (2, 3, 4)
warmup = 3
runs = 2
n_values = (4_000, 16_000)

runtime_rows = []
for n in n_values:
    stats = benchmark_runtime_case(
        n=n,
        theta=theta,
        leaf_size=leaf_size,
        max_order=max_order,
        warmup=warmup,
        runs=runs,
        dtype=jnp.float64,
        p_gears=p_gears,
        seed=0,
    )
    for label, values in stats.items():
        runtime_rows.append({'n': n, 'mode': label, **values})

runtime_rows


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

labels = [f"{row['mode']}\nN={row['n']}" for row in runtime_rows]
traversal = [row['traversal_s'] for row in runtime_rows]
total = [row['total_s'] for row in runtime_rows]

axes[0].bar(labels, traversal, color=['#7a9e7e' if row['mode'] == 'adaptive' else '#c9a66b' for row in runtime_rows])
axes[0].set_title('Traversal Time')
axes[0].set_ylabel('seconds')
axes[0].tick_params(axis='x', rotation=30)

axes[1].bar(labels, total, color=['#7a9e7e' if row['mode'] == 'adaptive' else '#c9a66b' for row in runtime_rows])
axes[1].set_title('Total Runtime')
axes[1].set_ylabel('seconds')
axes[1].tick_params(axis='x', rotation=30)

fig.tight_layout()
plt.show()

for row in runtime_rows:
    print(
        f"mode={row['mode']:<8} N={row['n']:<6d} tree={row['tree_s']:.3f}s "
        f"traversal={row['traversal_s']:.3f}s m2l={row['m2l_s']:.3f}s total={row['total_s']:.3f}s "
        f"far={row['far_pairs']:<8d} near={row['near_pairs']:<8d} gears={row['gear_counts']}"
    )


## Accuracy Diagnostics

For the accuracy comparison we use a smaller problem and compute a direct-sum
reference acceleration at every particle position.

The plots and summaries below are intentionally Dehnen-like:

- **relative acceleration error per particle**
- **distribution / cumulative fraction of errors**
- **percentile summaries** (`median`, `p90`, `p99`, `max`)

Here the relative error is

\[

rac{\|a_\mathrm{FMM} - a_\mathrm{ref}\|}{\|a_\mathrm{ref}\| + 
arepsilon}.
\]


In [ ]:
n_accuracy = 512
theta_accuracy = 0.5
leaf_size_accuracy = 16
softening = 1.0e-2
key = jax.random.PRNGKey(123)
positions_acc, masses_acc, _ = generate_random_distribution(
    n_accuracy,
    key=key,
    dtype=jnp.float64,
)

fixed_solver = make_solver(adaptive_order=False, theta=theta_accuracy, softening=softening)
adaptive_solver = make_solver(
    adaptive_order=True,
    theta=theta_accuracy,
    softening=softening,
    p_gears=p_gears,
)

ref_timing = time_callable(
    direct_sum_accelerations,
    positions_acc,
    masses_acc,
    softening=softening,
    warmup=1,
    runs=1,
)
acc_ref = ref_timing.result

fixed_timing = time_callable(
    lambda pos, mass: fixed_solver.compute_accelerations(pos, mass, leaf_size=leaf_size_accuracy, max_order=max_order),
    positions_acc,
    masses_acc,
    warmup=2,
    runs=2,
)
adaptive_timing = time_callable(
    lambda pos, mass: adaptive_solver.compute_accelerations(pos, mass, leaf_size=leaf_size_accuracy, max_order=max_order),
    positions_acc,
    masses_acc,
    warmup=2,
    runs=2,
)

acc_fixed = fixed_timing.result
acc_adaptive = adaptive_timing.result

rel_fixed, abs_fixed, ref_mag = relative_acceleration_errors(acc_fixed, acc_ref)
rel_adaptive, abs_adaptive, _ = relative_acceleration_errors(acc_adaptive, acc_ref)

summary_fixed = summarize_errors(rel_fixed)
summary_adaptive = summarize_errors(rel_adaptive)

print('direct sum runtime:', ref_timing.mean)
print('fixed runtime:', fixed_timing.mean)
print('adaptive runtime:', adaptive_timing.mean)
print('fixed summary:', summary_fixed)
print('adaptive summary:', summary_adaptive)
print('adaptive gear counts:', adaptive_solver._impl._recent_far_pairs_by_gear_counts)


In [ ]:
thresholds = (1e-4, 1e-3, 1e-2, 1e-1)
for label, rel in [('fixed', rel_fixed), ('adaptive', rel_adaptive)]:
    rel_np = np.asarray(rel)
    fractions = {thr: float(np.mean(rel_np <= thr)) for thr in thresholds}
    print(label, fractions)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

bins = np.logspace(-8, 0, 60)
axes[0].hist(np.asarray(rel_fixed), bins=bins, histtype='step', linewidth=2, label='fixed')
axes[0].hist(np.asarray(rel_adaptive), bins=bins, histtype='step', linewidth=2, label='adaptive')
axes[0].set_xscale('log')
axes[0].set_yscale('log')
axes[0].set_title('Relative Acceleration Error Histogram')
axes[0].set_xlabel('relative error')
axes[0].set_ylabel('count')
axes[0].legend()

x_fixed, y_fixed = empirical_cdf(rel_fixed)
x_adapt, y_adapt = empirical_cdf(rel_adaptive)
axes[1].plot(x_fixed, y_fixed, label='fixed', linewidth=2)
axes[1].plot(x_adapt, y_adapt, label='adaptive', linewidth=2)
axes[1].set_xscale('log')
axes[1].set_title('Cumulative Distribution of Relative Errors')
axes[1].set_xlabel('relative error')
axes[1].set_ylabel('fraction of particles')
axes[1].legend()

fig.tight_layout()
plt.show()


## Adaptive Error-Model Sweep

This sweep compares the validated default `tail_proxy` model against the opt-in
`dehnen_degree` estimator for a small set of `adaptive_eps` values. It reports
warmed runtime, Dehnen-style relative-acceleration error summaries, and gear counts.


In [ ]:
theta_sweep = 0.7
sweep_cases = [('tail_proxy', None), ('dehnen_degree', 0.005), ('dehnen_degree', 0.01), ('dehnen_degree', 0.02)]
sweep_rows = []
for error_model, adaptive_eps in sweep_cases:
    solver = make_solver(
        adaptive_order=True,
        theta=theta_sweep,
        softening=softening,
        p_gears=p_gears,
        adaptive_error_model=error_model,
        adaptive_eps=adaptive_eps,
        advanced=benchmark_runtime_config(),
    )
    runtime_timing = time_callable(
        lambda pos, mass: solver.compute_accelerations(pos, mass, leaf_size=leaf_size_accuracy, max_order=max_order),
        positions_acc,
        masses_acc,
        warmup=2,
        runs=2,
    )
    rel_err, _, _ = relative_acceleration_errors(runtime_timing.result, acc_ref)
    row = {
        'error_model': error_model,
        'adaptive_eps': adaptive_eps,
        'runtime_s': float(runtime_timing.mean),
        **summarize_errors(rel_err),
        'gear_counts': tuple(int(v) for v in solver._impl._recent_far_pairs_by_gear_counts),
    }
    sweep_rows.append(row)

sweep_rows


In [ ]:
labels = [f"{row['error_model']}\n eps={row['adaptive_eps']}" for row in sweep_rows]
runtime_vals = [row['runtime_s'] for row in sweep_rows]
p99_vals = [row['p99'] for row in sweep_rows]
fig, ax = plt.subplots(figsize=(7, 4))
ax.scatter(runtime_vals, p99_vals, s=80, color='#2f6c8f')
for label, x, y in zip(labels, runtime_vals, p99_vals):
    ax.annotate(label, (x, y), textcoords='offset points', xytext=(5, 5), fontsize=8)
ax.set_xlabel('runtime [s]')
ax.set_ylabel('p99 relative acceleration error')
ax.set_xscale('linear')
ax.set_yscale('log')
ax.set_title('Adaptive Error-Model Sweep: Runtime vs p99 Error')
fig.tight_layout()
plt.show()


## Estimator Comparison

This section compares the **current runtime proxy** against an **experimental degree-weighted Dehnen-like estimator** on the same source nodes and a simple range of opening factors. It does not change the runtime path; it is only a diagnostic to show how the two estimators behave.

In [ ]:
diag_solver = make_solver(adaptive_order=True, theta=0.5, p_gears=(2, 3, 4), advanced=benchmark_runtime_config())
diag_staged = build_tree_and_upward(
    diag_solver,
    positions_tune,
    masses_tune,
    leaf_size=tuning_leaf_size,
    max_order=tuning_max_order,
)
degree_power = source_power_by_degree_from_multipoles(
    multipole_packed=diag_staged.tree_artifacts.upward.multipoles.packed,
)
proxy_runtime = source_error_proxy_by_order_from_degree_power(
    degree_power=degree_power,
    p_gears=(2, 3, 4),
)
opening_grid = jnp.asarray([0.2, 0.35, 0.5, 0.65], dtype=proxy_runtime.dtype)
estimator_rows = []
for opening in opening_grid:
    est = dehnen_like_pair_error_by_order_from_degree_power(
        degree_power=degree_power[:64],
        opening=jnp.full((64,), opening, dtype=proxy_runtime.dtype),
        order_values=jnp.asarray((2, 3, 4), dtype=jnp.int32),
    )
    estimator_rows.append((float(opening), np.mean(np.asarray(est), axis=0)))

print('runtime proxy mean by gear:', np.mean(np.asarray(proxy_runtime[:64]), axis=0))
for opening, values in estimator_rows:
    print(f'opening={opening:.2f} dehnen-like mean by gear={values}')


In [ ]:
runtime_mean = np.mean(np.asarray(proxy_runtime[:64]), axis=0)
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot((2, 3, 4), runtime_mean, marker='o', linewidth=2, label='current runtime proxy')
for opening, values in estimator_rows:
    ax.plot((2, 3, 4), values, marker='o', linestyle='--', label=f'dehnen-like, opening={opening:.2f}')
ax.set_yscale('log')
ax.set_xlabel('order')
ax.set_ylabel('mean estimated tail/error scale')
ax.set_title('Current Proxy vs Experimental Degree-Weighted Estimator')
ax.legend(loc='best')
fig.tight_layout()
plt.show()


## Interpreting the Results

A useful reading of this notebook is:

- **Performance section**:
  compare whether adaptive acceptance reduces traversal and/or total time relative
  to the fixed-order baseline.
- **Accuracy section**:
  compare whether adaptive acceptance preserves the same relative acceleration-error
  distribution shape and percentiles.

If adaptive is working well, you should typically see:

- lower or comparable total runtime,
- a shifted gear distribution instead of everything landing in the highest order,
- relative error percentiles that remain close to the fixed-order baseline.

If adaptive starts over-accepting, the histogram and percentile diagnostics will
usually show it immediately in the high-error tail.
